# Build the map-ready tables from `spend-patterns-rice`

This notebook does one job. It opens the cached `spend-patterns-rice.parquet`
sitting in your Drive, keeps only the energy / food / water businesses, and
writes two small files back to Drive:

- **`efw_places.parquet`** — one row per shop, with a latitude and longitude.
  This draws the dots on the map.
- **`efw_daily_city.parquet`** — one row per city, per type, per day.
  This drives the time slider, and later the matchday work.

Why this dataset and not `store-visits-rice`: `spend-patterns-rice` has real
coordinates on every row, has a `PLACEKEY` that links cleanly to
`core-poi-geometry-rice`, keeps Dallas and Houston as separate labels instead
of one merged `Dallas / Houston`, and holds a day-by-day spend breakdown.
`store-visits-rice` has none of that.

Run the cells top to bottom. Takes a couple of minutes.

> Sample data were transformed by the organisers (noise on magnitudes, spatial
> jitter). Everything here demonstrates methodology, not real city rankings.

In [ ]:
# ============================================================
# STEP 0 — mount Drive and point at the cache.
# Same cache folder that explore_raw.ipynb writes to.
# ============================================================
import os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/ricehack_cache"
except ImportError:
    CACHE = os.path.expanduser("~/ricehack_cache")

SRC = f"{CACHE}/spend-patterns-rice.parquet"
print("reading from:", SRC)
print("exists:", os.path.exists(SRC))

## The settings, all in one place

Three things to know before you change anything here.

`EFW_NAICS3` is the list of business-type codes we keep. NAICS is a government
code for what kind of business something is, and we only use the first three
digits: `722` means restaurants, `447` means gas stations, `721` means hotels.
Everything not in this list gets thrown away.

`layer_of` sorts those codes into the four buckets the map colours by. It
matches the mapping already used in `scripts/build_track2_curated_package.py`,
so the two agree.

`CENTRES` is the fix for a real problem. `MARKET` is a label, not a map
boundary, so a row tagged Atlanta can have coordinates in Orlando. We drop
anything more than `RADIUS_KM` from its own city's centre.

In [ ]:
# Business types that count as energy / food / water.
EFW_NAICS3 = {
    "722",  # food service
    "445",  # food and beverage stores
    "447",  # gasoline
    "721",  # lodging
    "711",  # performing arts and sports
    "712",  # museums
    "713",  # amusement
    "562",  # waste
    "221",  # utilities
    "311",  # food manufacturing
    "312",  # beverage manufacturing
    "424",  # grocery wholesale
    "485",  # transit
    "481",  # air transport
    "488",  # transport support
}

def layer_of(n3):
    if n3 in {"722", "445", "311", "312", "424"}: return "Food"
    if n3 in {"447", "221"}:                     return "Energy"
    if n3 in {"721", "562"}:                     return "Water"
    if n3 in {"711", "712", "713"}:              return "Venue"
    return "Other_EFW"

# Host city centres, used only to throw out stray coordinates.
CENTRES = {
    "Atlanta":                (33.749,  -84.388),
    "Boston":                 (42.360,  -71.058),
    "Dallas":                 (32.777,  -96.797),
    "Houston":                (29.760,  -95.370),
    "Kansas City":            (39.100,  -94.579),
    "Los Angeles":            (34.052, -118.244),
    "San Francisco Bay Area": (37.775, -122.419),
    "Miami":                  (25.775,  -80.194),
    "New York/New Jersey":    (40.713,  -74.006),
    "Philadelphia":           (39.953,  -75.165),
    "Seattle":                (47.606, -122.332),
}
RADIUS_KM = 75

# Only these columns get read. The file has about 48; we need 18.
COLS = [
    "PLACEKEY", "LOCATION_NAME", "MARKET", "CITY", "REGION", "POSTAL_CODE",
    "LATITUDE", "LONGITUDE", "NAICS_CODE", "TOP_CATEGORY", "SUB_CATEGORY",
    "BRANDS", "SPEND_DATE_RANGE_START", "RAW_NUM_CUSTOMERS",
    "RAW_NUM_TRANSACTIONS", "RAW_TOTAL_SPEND", "SPEND_BY_DAY",
    "SPEND_PER_TRANSACTION_BY_DAY",
]

## Step 1 — read, filter, clean

One row of this file is one shop for one month. Reading only the columns we
need keeps 831 MB on disk down to something Colab handles without complaint.

In [ ]:
print("reading, this takes about a minute...")
df = pq.read_table(SRC, columns=COLS).to_pandas()
raw_rows = len(df)
print(f"{raw_rows:,} shop-months")
print("markets found:", sorted(df.MARKET.dropna().unique()))

# Keep only energy / food / water businesses.
df["NAICS3"] = df["NAICS_CODE"].astype(str).str[:3]
df = df[df["NAICS3"].isin(EFW_NAICS3)].copy()
df["efw_layer"] = df["NAICS3"].map(layer_of)
after_efw = len(df)

# Drop coordinates that fall outside their own city.
lat0 = df["MARKET"].map(lambda m: CENTRES.get(m, (np.nan, np.nan))[0])
lon0 = df["MARKET"].map(lambda m: CENTRES.get(m, (np.nan, np.nan))[1])
dist_km = np.sqrt(
    ((df.LATITUDE - lat0) * 111) ** 2
    + ((df.LONGITUDE - lon0) * 111 * np.cos(np.radians(lat0))) ** 2
)
outside = (dist_km > RADIUS_KM) | dist_km.isna()
n_outside = int(outside.sum())
df = df[~outside].copy()

df["month"] = pd.to_datetime(df["SPEND_DATE_RANGE_START"]).dt.to_period("M").astype(str)
print(f"kept {after_efw:,} EFW rows, then dropped {n_outside:,} for bad coordinates")
print(f"{len(df):,} rows left")

## Step 2 — one row per shop

This is the map's base layer. Every row becomes one dot, placed by its
latitude and longitude, coloured by `efw_layer`, and sized by however many
customers it saw.

In [ ]:
places = (
    df.groupby("PLACEKEY", as_index=False).agg(
        LOCATION_NAME=("LOCATION_NAME", "first"),
        MARKET=("MARKET", "first"),
        CITY=("CITY", "first"),
        REGION=("REGION", "first"),
        POSTAL_CODE=("POSTAL_CODE", "first"),
        LATITUDE=("LATITUDE", "first"),
        LONGITUDE=("LONGITUDE", "first"),
        efw_layer=("efw_layer", "first"),
        NAICS3=("NAICS3", "first"),
        TOP_CATEGORY=("TOP_CATEGORY", "first"),
        total_customers=("RAW_NUM_CUSTOMERS", "sum"),
        total_transactions=("RAW_NUM_TRANSACTIONS", "sum"),
        total_spend=("RAW_TOTAL_SPEND", "sum"),
        months_seen=("month", "nunique"),
    )
)
places.to_parquet(f"{CACHE}/efw_places.parquet", index=False)
print(f"wrote efw_places.parquet — {len(places):,} shops")
places.head()

## Step 2.5 — is the division method legit?

There are two ways to get a daily demand number out of this file, and this
section works out which one to trust.

**The proportional way.** `RAW_NUM_CUSTOMERS` is the month's customer count,
one number for the whole month. Split it across the month's days in proportion
to how much money came in each day. This assumes every customer spends the same
on a quiet Tuesday as on a busy Saturday, which is a guess.

**The division way.** `SPEND_BY_DAY` is money taken each day, and
`SPEND_PER_TRANSACTION_BY_DAY` is the average size of a purchase each day.
Divide one by the other and you get the actual number of purchases on that day.
Nothing is assumed, it is arithmetic.

The test below does not just compare the two against each other, because that
only tells you they disagree, not which one is right. It checks each one against
a number the file already states. If the daily purchase counts add up to
`RAW_NUM_TRANSACTIONS` for the month, the division method is sound.

In [ ]:
# ============================================================
# Sanity check. Runs on a sample, so it is quick.
# ============================================================
SAMPLE_N = 3000
chk = df.sample(min(SAMPLE_N, len(df)), random_state=0).copy()

def as_vals(v):
    """Variant columns arrive as a list, a dict, or JSON text of either."""
    if isinstance(v, dict):
        return [v[k] for k in sorted(v)]
    if isinstance(v, (list, np.ndarray)):
        return list(v)
    if isinstance(v, str):
        try:
            p = json.loads(v)
        except Exception:
            return []
        return [p[k] for k in sorted(p)] if isinstance(p, dict) else list(p)
    return []

print("SPEND_BY_DAY               :", str(chk.SPEND_BY_DAY.iloc[0])[:100])
print("SPEND_PER_TRANSACTION_BY_DAY:", str(chk.SPEND_PER_TRANSACTION_BY_DAY.iloc[0])[:100])
print()

rows, pairs = [], []
for r in chk.itertuples(index=False):
    spend_d = np.array(as_vals(r.SPEND_BY_DAY), dtype="float64")
    price_d = np.array(as_vals(r.SPEND_PER_TRANSACTION_BY_DAY), dtype="float64")
    if len(spend_d) == 0 or len(spend_d) != len(price_d):
        continue
    if not (r.RAW_TOTAL_SPEND > 0 and r.RAW_NUM_TRANSACTIONS > 0 and r.RAW_NUM_CUSTOMERS > 0):
        continue

    # division way: purchases per day, then people per day
    with np.errstate(divide="ignore", invalid="ignore"):
        txn_d = np.where(price_d > 0, spend_d / price_d, np.nan)
    cust_per_txn = r.RAW_NUM_CUSTOMERS / r.RAW_NUM_TRANSACTIONS
    cust_div = txn_d * cust_per_txn

    # proportional way: people per day
    cust_prop = spend_d / r.RAW_TOTAL_SPEND * r.RAW_NUM_CUSTOMERS

    rows.append({
        "spend_gap_pct": (spend_d.sum() - r.RAW_TOTAL_SPEND) / r.RAW_TOTAL_SPEND * 100,
        "txn_gap_pct":   (np.nansum(txn_d) - r.RAW_NUM_TRANSACTIONS) / r.RAW_NUM_TRANSACTIONS * 100,
        "layer": r.efw_layer,
    })
    ok = (cust_prop > 0) & np.isfinite(cust_div)
    pairs.append(np.abs(cust_div[ok] - cust_prop[ok]) / cust_prop[ok] * 100)

res = pd.DataFrame(rows)
dev = np.concatenate(pairs) if pairs else np.array([])
print(f"checked {len(res):,} shop-months, {len(dev):,} individual days\n")

In [ ]:
# ============================================================
# The verdict, in plain english.
# ============================================================
def band(x):
    return f"median {np.nanmedian(x):+.2f}%   half fall within {np.nanpercentile(np.abs(x),75):.2f}%   worst 5% beyond {np.nanpercentile(np.abs(x),95):.2f}%"

print("TEST 1 — does the daily money add up to the stated monthly total?")
print("  ", band(res.spend_gap_pct))
print("   near zero means SPEND_BY_DAY is trustworthy on its own.")
print()
print("TEST 2 — do the divided-out daily purchases add up to RAW_NUM_TRANSACTIONS?")
print("  ", band(res.txn_gap_pct))
print("   near zero means the division method recovers a number the file already knows,")
print("   which is the whole case for using it.")
print()
print("TEST 3 — how far apart are the two methods on any given day?")
print(f"   median difference {np.median(dev):.1f}%,  a quarter of days differ by more than {np.percentile(dev,75):.1f}%,")
print(f"   and the worst 5% of days differ by more than {np.percentile(dev,95):.1f}%.")
print("   large numbers here are not a fault, they are the proportional method's error showing.")
print()

print("Test 2 broken down by business type (this is the one that matters):")
display(res.groupby("layer").txn_gap_pct.agg(
    shop_months="size",
    median_gap_pct=lambda s: round(np.nanmedian(s), 2),
    within_pct_75=lambda s: round(np.nanpercentile(np.abs(s), 75), 2),
))

verdict = np.nanmedian(np.abs(res.txn_gap_pct))
print()
if verdict < 2:
    print(f"VERDICT: division method is sound. It reproduces the stated monthly")
    print(f"transaction count to within {verdict:.2f}% for a typical shop-month. Use it.")
elif verdict < 10:
    print(f"VERDICT: division method is usable but loose, off by about {verdict:.1f}%.")
    print("Fine for showing shape over time, weak for absolute totals.")
else:
    print(f"VERDICT: division method does not reconcile, off by {verdict:.1f}%.")
    print("Something about the two day columns does not line up. Show me this output.")

### Is a single monthly scale factor good enough?

Fair question, and this cell answers it rather than assuming it.

The scale factor is how much the divided-out daily purchase counts have to be
shrunk so they add up to the month's stated `RAW_NUM_TRANSACTIONS`. Rescaling
removes the average bias, but it cannot fix the day-to-day part: if the gap
between the typical purchase and the average purchase swings hard from one day
to the next, some days end up too high and others too low.

So the test is whether the same shop needs roughly the same factor every month.
If one shop lands on 0.92, then 0.91, then 0.93, the skew is a stable trait of
that shop's customers and almost certainly steady within a month too. If it
jumps around, the assumption is weak and belongs in a footnote on the chart.

Bear in mind the floor. Test 1 showed the organisers' injected noise is about
11%, so nothing here can be more accurate than that.

In [ ]:
# ============================================================
# How stable is the scale factor?
# ============================================================
fac = []
for r in chk.itertuples(index=False):
    s = np.array(as_vals(r.SPEND_BY_DAY), dtype="float64")
    p = np.array(as_vals(r.SPEND_PER_TRANSACTION_BY_DAY), dtype="float64")
    if len(s) == 0 or len(s) != len(p) or r.RAW_NUM_TRANSACTIONS <= 0:
        continue
    t = np.where(p > 0, s / p, 0.0).sum()
    if t > 0:
        fac.append({"PLACEKEY": r.PLACEKEY, "layer": r.efw_layer,
                    "month": r.month, "scale": r.RAW_NUM_TRANSACTIONS / t})
fac = pd.DataFrame(fac)

print(f"scale factors from {len(fac):,} shop-months\n")
print(f"typical factor          : {fac.scale.median():.3f}")
print(f"middle half sit between : {fac.scale.quantile(.25):.3f} and {fac.scale.quantile(.75):.3f}")
print(f"spread across all shops : {fac.scale.std() / fac.scale.mean() * 100:.1f}%")
print()

# The real test: same shop, different months. Does its factor hold still?
rep = fac.groupby("PLACEKEY").filter(lambda g: len(g) >= 3)
if len(rep):
    wobble = rep.groupby("PLACEKEY").scale.agg(lambda s: s.std() / s.mean() * 100)
    print(f"Shops seen in 3+ months: {wobble.size:,}")
    print(f"How much one shop's factor moves month to month:")
    print(f"   typical {wobble.median():.1f}%,  three quarters under {wobble.quantile(.75):.1f}%,"
          f"  worst 5% over {wobble.quantile(.95):.1f}%")
    print()
    w = wobble.median()
    if w < 5:
        print(f"VERDICT: very stable. A shop's skew barely moves ({w:.1f}%), which is well")
        print("inside the 11% noise floor, so one factor per month is fine.")
    elif w < 12:
        print(f"VERDICT: stable enough. A shop's factor moves about {w:.1f}%, which is at or")
        print("under the noise the organisers injected, so you cannot do better anyway.")
    else:
        print(f"VERDICT: shaky. A shop's factor moves {w:.1f}% between months, above the noise")
        print("floor, so say on the chart that daily numbers carry roughly this much error.")
else:
    print("not enough repeat shops in the sample; raise SAMPLE_N")

display(fac.groupby("layer").scale.agg(
    shop_months="size", typical=lambda s: round(s.median(), 3),
    spread_pct=lambda s: round(s.std() / s.mean() * 100, 1)))

In [ ]:
# One shop, both methods drawn on top of each other, so you can see the shape.
import matplotlib.pyplot as plt

r = chk[(chk.RAW_TOTAL_SPEND > 0) & (chk.RAW_NUM_TRANSACTIONS > 0)].iloc[0]
spend_d = np.array(as_vals(r.SPEND_BY_DAY), dtype="float64")
price_d = np.array(as_vals(r.SPEND_PER_TRANSACTION_BY_DAY), dtype="float64")
txn_d   = np.where(price_d > 0, spend_d / price_d, np.nan)

div  = txn_d * (r.RAW_NUM_CUSTOMERS / r.RAW_NUM_TRANSACTIONS)
prop = spend_d / r.RAW_TOTAL_SPEND * r.RAW_NUM_CUSTOMERS

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(div,  label="division method (from real daily purchase counts)", lw=2)
ax.plot(prop, label="proportional method (split by daily spend)", lw=2, ls="--")
ax.set_title(f"{r.LOCATION_NAME} — {r.MARKET} — {r.month}")
ax.set_xlabel("day of month"); ax.set_ylabel("customers")
ax.legend(); plt.show()

## Step 3 — one row per city, per type, per day

This is where the daily demand curve comes from, and it is built in three
moves.

First, **purchases per day**. `SPEND_BY_DAY` is the money taken each day and
`SPEND_PER_TRANSACTION_BY_DAY` is the typical size of a purchase that day, so
dividing one by the other gives the number of purchases. That is arithmetic on
two measured columns, not a guess.

Second, **rescale so the month adds up**. Step 2.5 showed those divided-out
counts run about 9% high, almost certainly because the price column is a middle
value rather than an average, and the average sits higher because a few large
purchases drag it up. Since the file already states the month's true
`RAW_NUM_TRANSACTIONS`, we shrink each month's days by whatever single factor
makes them add up to it. The daily shape survives, the bias goes.

Third, **purchases to people**. Multiply by the month's ratio of
`RAW_NUM_CUSTOMERS` to `RAW_NUM_TRANSACTIONS`, roughly how many purchases one
customer makes. After this the days add up exactly to the stated monthly
customer count.

What is still assumed: that the skew between typical and average purchase holds
steady across the days of a month. Step 2.5 measures how safe that is. What is
measured outright: daily spend, daily purchase counts, and both monthly totals.

We aggregate to city rather than shop on purpose. Shop by day would be roughly
40 million rows, too heavy for Colab and for a browser, whereas city by type by
day is about a hundred thousand.

In [ ]:
daily_parts = []
n_skipped = 0

for mkt, g in df.groupby("MARKET", sort=False):
    s_arr = g["SPEND_BY_DAY"].map(as_vals)
    p_arr = g["SPEND_PER_TRANSACTION_BY_DAY"].map(as_vals)

    # keep only rows where the two day columns line up and totals are usable
    keep = (
        (s_arr.map(len) > 0)
        & (s_arr.map(len) == p_arr.map(len))
        & (g.RAW_NUM_TRANSACTIONS.values > 0)
        & (g.RAW_NUM_CUSTOMERS.values > 0)
    )
    n_skipped += int((~keep).sum())
    g, s_arr, p_arr = g[keep], s_arr[keep], p_arr[keep]
    if len(g) == 0:
        print(f"  {mkt}: nothing usable, skipped")
        continue

    n_days = s_arr.map(len).values
    flat_s = np.concatenate(s_arr.values).astype("float64")
    flat_p = np.concatenate(p_arr.values).astype("float64")

    # 1. purchases per day, straight from the two measured columns
    txn_raw = np.where(flat_p > 0, flat_s / flat_p, 0.0)

    # 2. rescale each shop-month so its days sum to the stated total
    offsets = np.concatenate([[0], np.cumsum(n_days)[:-1]])
    row_sum = np.add.reduceat(txn_raw, offsets)
    scale = np.where(row_sum > 0, g.RAW_NUM_TRANSACTIONS.values / row_sum, 0.0)
    txn = txn_raw * np.repeat(scale, n_days)

    # 3. purchases to people
    cust_per_txn = g.RAW_NUM_CUSTOMERS.values / g.RAW_NUM_TRANSACTIONS.values
    cust = txn * np.repeat(cust_per_txn, n_days)

    out = pd.DataFrame({
        "MARKET": mkt,
        "efw_layer": np.repeat(g["efw_layer"].values, n_days),
        "date": np.concatenate([
            pd.date_range(s, periods=n, freq="D").values
            for s, n in zip(pd.to_datetime(g.SPEND_DATE_RANGE_START), n_days)
        ]),
        "spend": flat_s,
        "transactions": txn,
        "customers": cust,
    })

    daily_parts.append(
        out.groupby(["MARKET", "efw_layer", "date"], as_index=False).agg(
            spend=("spend", "sum"),
            transactions=("transactions", "sum"),
            customers=("customers", "sum"),
        )
    )
    print(f"  unpacked {mkt}")

daily = pd.concat(daily_parts, ignore_index=True)
daily.to_parquet(f"{CACHE}/efw_daily_city.parquet", index=False)
print(f"\nwrote efw_daily_city.parquet — {len(daily):,} rows")
print(f"skipped {n_skipped:,} shop-months whose day columns did not line up")
daily.head()

In [ ]:
# Proof the rescaling landed: monthly totals rebuilt from the daily file
# should match the monthly totals stated in the source file.
reb = (daily.assign(month=daily.date.dt.to_period("M").astype(str))
            .groupby("month")[["transactions", "customers"]].sum())
src = (df[df.RAW_NUM_TRANSACTIONS > 0]
       .groupby("month")[["RAW_NUM_TRANSACTIONS", "RAW_NUM_CUSTOMERS"]].sum())
cmp = reb.join(src)
cmp["txn_gap_pct"]  = (cmp.transactions - cmp.RAW_NUM_TRANSACTIONS) / cmp.RAW_NUM_TRANSACTIONS * 100
cmp["cust_gap_pct"] = (cmp.customers - cmp.RAW_NUM_CUSTOMERS) / cmp.RAW_NUM_CUSTOMERS * 100

print("Rebuilt monthly totals vs the totals the file states:")
print(f"  purchases: worst month is off by {cmp.txn_gap_pct.abs().max():.4f}%")
print(f"  customers: worst month is off by {cmp.cust_gap_pct.abs().max():.4f}%")
print("(these should be essentially zero — the rescaling forces it.")
print(" any gap is the shop-months skipped above, not the method.)")
display(cmp.head())

## Step 4 — sanity check

Read these numbers before you trust anything downstream. If a city is missing,
or a shop count looks absurd, the problem is here and not in the map.

In [ ]:
print("=" * 60)
print(f"Started with {raw_rows:,} rows of shop-months.")
print(f"Kept {after_efw:,} after keeping only energy/food/water businesses.")
print(f"Threw away {n_outside:,} rows whose coordinates were nowhere near their own city.")
print()
print(f"Found {places.MARKET.nunique()} host cities and {len(places):,} individual shops.")
print(f"Daily numbers run from {daily.date.min().date()} to {daily.date.max().date()}.")
print(f"Total card customers across everything: {places.total_customers.sum():,.0f}")
print(f"Total spend across everything: ${places.total_spend.sum():,.0f}")
print()
print("Shops and customers per city:")
display(
    places.groupby("MARKET")
    .agg(shops=("PLACEKEY", "count"), customers=("total_customers", "sum"))
    .sort_values("shops", ascending=False)
)
print("Shops per city, broken down by type:")
display(pd.crosstab(places.MARKET, places.efw_layer))

In [ ]:
# A first look at the map, just to confirm the dots land in the right places.
# Not the real map, only a check that nothing is sitting in the Pacific.
import matplotlib.pyplot as plt

COLOURS = {"Food": "#d97706", "Energy": "#2563eb", "Water": "#0891b2",
           "Venue": "#dc2626", "Other_EFW": "#9ca3af"}

fig, ax = plt.subplots(figsize=(11, 6))
for layer, g in places.groupby("efw_layer"):
    ax.scatter(g.LONGITUDE, g.LATITUDE, s=1, alpha=0.4,
               c=COLOURS.get(layer, "#999"), label=layer)
ax.set_title("EFW shops across the host cities")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(markerscale=8, loc="lower left")
plt.show()

## Step 5 — the heat layer

`urban-heat-index-rice` is the only dataset that changes *within* a city, so it
is what lets the map say which neighbourhood is hot rather than only which city
is hot. `UHI` is an index from 1 to 11, not a temperature, so it supports
"this district scores higher than that one" and never "this district is four
degrees warmer".

Two problems to fix. It carries the same bad labels as everything else, with
coordinates running from latitude 18 to 61, meaning Hawaii to Alaska, so the
same city-centre filter applies. And 1.2 million points will choke a browser
and look like confetti anyway.

The fix for both is to round coordinates onto a grid and average the heat inside
each square. At `GRID_DEG = 0.01` a square is roughly 1.1 km across, which is
about a neighbourhood, and it turns a million points into a few thousand tiles
that draw instantly and read as a smooth heat blanket.

The last cell attaches a heat score to every shop by looking up which tile it
falls in. That is the join that makes the whole thing work, because it lets you
ask which restaurants sit in the hottest parts of Houston.

In [ ]:
# ============================================================
# STEP 5 — grid the urban heat index down to map size
# ============================================================
GRID_DEG = 0.01   # about 1.1 km. raise to 0.02 for fewer, chunkier tiles.

uhi = pd.read_parquet(f"{CACHE}/urban-heat-index-rice.parquet")
print("columns:", list(uhi.columns))
uhi_raw = len(uhi)
print(f"{uhi_raw:,} heat points")
print(f"latitude runs {uhi.LATITUDE.min():.1f} to {uhi.LATITUDE.max():.1f} "
      "(Hawaii to Alaska — the labels are not boundaries)")

uhi["UHI"] = pd.to_numeric(uhi["UHI"], errors="coerce")
uhi = uhi.dropna(subset=["LATITUDE", "LONGITUDE", "UHI"])

# Same city-centre filter used on the shops.
lat0 = uhi["MARKET"].map(lambda m: CENTRES.get(m, (np.nan, np.nan))[0])
lon0 = uhi["MARKET"].map(lambda m: CENTRES.get(m, (np.nan, np.nan))[1])
d_km = np.sqrt(
    ((uhi.LATITUDE - lat0) * 111) ** 2
    + ((uhi.LONGITUDE - lon0) * 111 * np.cos(np.radians(lat0))) ** 2
)
outside = (d_km > RADIUS_KM) | d_km.isna()
print(f"dropping {int(outside.sum()):,} points that sit nowhere near their own city")
uhi = uhi[~outside].copy()

# Round onto a grid and average the heat inside each square.
uhi["lat_bin"] = (uhi.LATITUDE  / GRID_DEG).round() * GRID_DEG
uhi["lon_bin"] = (uhi.LONGITUDE / GRID_DEG).round() * GRID_DEG

heat = (uhi.groupby(["MARKET", "lat_bin", "lon_bin"], as_index=False)
           .agg(uhi_mean=("UHI", "mean"),
                uhi_max=("UHI", "max"),
                n_points=("UHI", "size")))
heat[["lat_bin", "lon_bin"]] = heat[["lat_bin", "lon_bin"]].round(4)
heat["uhi_mean"] = heat.uhi_mean.round(2)

heat.to_parquet(f"{CACHE}/efw_heat_grid.parquet", index=False)
print(f"\nwrote efw_heat_grid.parquet — {uhi_raw:,} points became {len(heat):,} tiles")
print(f"that is {uhi_raw / max(len(heat), 1):.0f}x smaller\n")

display(heat.groupby("MARKET").agg(
    tiles=("uhi_mean", "size"),
    typical_heat=("uhi_mean", "median"),
    hottest_tile=("uhi_max", "max"),
).sort_values("typical_heat", ascending=False).round(2))

In [ ]:
# ============================================================
# Give every shop a heat score, by looking up the tile it sits in.
# This is the join that lets you ask "which restaurants are in the hot bits".
# ============================================================
places["lat_bin"] = (places.LATITUDE  / GRID_DEG).round() * GRID_DEG
places["lon_bin"] = (places.LONGITUDE / GRID_DEG).round() * GRID_DEG
places[["lat_bin", "lon_bin"]] = places[["lat_bin", "lon_bin"]].round(4)

places = places.merge(
    heat[["MARKET", "lat_bin", "lon_bin", "uhi_mean"]],
    on=["MARKET", "lat_bin", "lon_bin"], how="left",
).rename(columns={"uhi_mean": "uhi"})

hit = places.uhi.notna().mean() * 100
print(f"{hit:.1f}% of shops landed in a heat tile")
if hit < 60:
    print("low. the heat grid is sparser than the shops, so widen GRID_DEG to 0.02")
    print("and re-run both cells, or fall back to each city's median heat.")

# Fill the misses with their own city's typical heat, and flag which are guessed.
places["uhi_is_filled"] = places.uhi.isna()
places["uhi"] = places.uhi.fillna(places.groupby("MARKET").uhi.transform("median"))

places.to_parquet(f"{CACHE}/efw_places.parquet", index=False)
print(f"rewrote efw_places.parquet with a uhi column\n")

print("Hottest shops in each city, and how the types split:")
display(places.groupby("MARKET").agg(
    shops=("PLACEKEY", "size"),
    typical_heat=("uhi", "median"),
    shops_in_hot_spots=("uhi", lambda s: int((s >= 8).sum())),
).sort_values("typical_heat", ascending=False).round(2))

In [ ]:
# Eyeball one city: heat underneath, shops on top.
CITY = "Houston"

h = heat[heat.MARKET == CITY]
p = places[places.MARKET == CITY]

fig, ax = plt.subplots(figsize=(9, 8))
sc = ax.scatter(h.lon_bin, h.lat_bin, c=h.uhi_mean, s=14, cmap="inferno",
                marker="s", alpha=0.85)
ax.scatter(p.LONGITUDE, p.LATITUDE, s=3, c="#00e5ff", alpha=0.5, label="EFW shops")
plt.colorbar(sc, ax=ax, label="urban heat index (1-11)")
ax.set_title(f"{CITY} — heat tiles with EFW shops on top")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(markerscale=4, loc="lower left")
plt.show()

## Step 6 — export for the web page

`app/index.html` reads these files. This writes them into a `web/` folder in
your Drive so you can download them into `app/data/`.

The one that makes the map move through time is `sm/<city>.json`: for every shop
in that city, a list of 61 numbers, one per month, being that shop's customers
that month. That is the native grain of `spend-patterns-rice`, so it costs
nothing to produce. Shop-by-day would be roughly 40 million numbers and would
kill the browser, which is why the map animates by month while the chart shows
days inside whichever month you land on.

Short keys keep the files small: `m` market, `l` layer, `k` placekey, `n` name,
`x` longitude, `y` latitude, `c` customers, `u` heat index, `d` date, `s` spend.

In [ ]:
# ============================================================
# STEP 6 — write the files app/index.html expects
# ============================================================
import os, json

WEB = f"{CACHE}/web"
os.makedirs(f"{WEB}/sm", exist_ok=True)

MONTHS = sorted(df.month.unique())
json.dump(MONTHS, open(f"{WEB}/months.json", "w"))

# --- shops: one object per dot ------------------------------
places_sorted = places.sort_values(["MARKET", "PLACEKEY"]).reset_index(drop=True)
pd.DataFrame({
    "m": places_sorted.MARKET,
    "k": places_sorted.PLACEKEY,
    "n": places_sorted.LOCATION_NAME.fillna("(unnamed)"),
    "l": places_sorted.efw_layer,
    "x": places_sorted.LONGITUDE.round(5),
    "y": places_sorted.LATITUDE.round(5),
    "c": places_sorted.total_customers.round(0),
    "u": (places_sorted.uhi.round(1) if "uhi" in places_sorted else np.nan),
}).to_json(f"{WEB}/places.json", orient="records")

# --- heat tiles ---------------------------------------------
pd.DataFrame({
    "m": heat.MARKET, "x": heat.lon_bin.round(5),
    "y": heat.lat_bin.round(5), "u": heat.uhi_mean.round(1),
}).to_json(f"{WEB}/heat.json", orient="records")

# --- daily city totals, for the chart -----------------------
pd.DataFrame({
    "m": daily.MARKET, "l": daily.efw_layer,
    "d": daily.date.dt.strftime("%Y-%m-%d"),
    "c": daily.customers.round(1), "s": daily.spend.round(0),
}).to_json(f"{WEB}/daily.json", orient="records")

# --- shop x month, one file per city — this is what animates the map
mi = {m: i for i, m in enumerate(MONTHS)}
for city, g in df.groupby("MARKET"):
    keys = sorted(places_sorted.loc[places_sorted.MARKET == city, "PLACEKEY"])
    ki = {k: i for i, k in enumerate(keys)}
    v = np.zeros((len(keys), len(MONTHS)), dtype="float32")
    sub = g[g.PLACEKEY.isin(ki)]
    v[sub.PLACEKEY.map(ki).values, sub.month.map(mi).values] = sub.RAW_NUM_CUSTOMERS.values
    json.dump({"keys": keys, "v": np.round(v).astype(int).tolist()},
              open(f"{WEB}/sm/{city.replace('/', '_')}.json", "w"))
    print(f"  sm/{city}: {len(keys):,} shops x {len(MONTHS)} months")

print()
for root, _, files in os.walk(WEB):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, WEB):28s} {os.path.getsize(p)/1e6:6.1f} MB")

print(f"\nwritten to {WEB}")
print("download the whole web folder into app/data/, keeping the sm/ subfolder,")
print("then from the app folder run:  python -m http.server 8000")

## What comes next

Two files now sit in `ricehack_cache`, both small enough to hand to a browser.
`efw_places.parquet` becomes the dots, and `efw_daily_city.parquet` becomes
the time slider.

The heat layer still needs doing separately. `urban-heat-index-rice.parquet`
holds 1.2 million points, which will choke a browser, so round the
coordinates to two decimal places and average the heat value inside each
square. That turns confetti into a smooth heat blanket and cuts it to a few
thousand squares per city.